In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [3]:
spark = SparkSession.builder\
    .master("local[8]")\
    .appName("Local_UserBehavior")\
    .config("spark.driver.memory", "4g")\
    .config("spark.sql.shuffle.partitions", "4")\
    .getOrCreate()

26/05/06 20:03:27 WARN Utils: Your hostname, LAPTOP-KVJ76CML resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/06 20:03:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 20:03:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
spark.sql("use ods")
spark.sql("show tables").show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|      ods|ods_user_behavior|      false|
+---------+-----------------+-----------+



In [10]:
df_dates = spark.table("ods.ods_user_behavior").select("event_date").distinct()\
.withColumn("date_key", F.date_format("event_date","yyyyMMdd").cast("int"))\
.withColumn("year", F.year("event_date")) \
.withColumn("month", F.month("event_date")) \
.withColumn("day", F.dayofmonth("event_date"))

In [11]:
spark.sql("CREATE DATABASE IF NOT EXISTS dwd")

26/05/05 17:24:47 WARN ObjectStore: Failed to get database dwd, returning NoSuchObjectException
26/05/05 17:24:47 WARN ObjectStore: Failed to get database dwd, returning NoSuchObjectException
26/05/05 17:24:47 WARN ObjectStore: Failed to get database dwd, returning NoSuchObjectException


DataFrame[]

In [ ]:
df_dates.write.mode("overwrite").format("parquet").saveAsTable("dwd.dim_date")

NameError: name 'df_dates' is not defined

In [4]:
df_user = spark.table("ods.ods_user_behavior").select("user_id").distinct()
df_user.write.mode("overwrite").bucketBy(8,"user_id").sortBy("user_id")\
.format("parquet").saveAsTable("dwd.dim_user")

26/05/06 17:19:46 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

use 8 bucket because of 8 cores

In [6]:
df_item = spark.table("ods.ods_user_behavior").select("item_id", "category_id").distinct()
df_item.write.mode("overwrite").bucketBy(8, "item_id").sortBy("item_id") \
    .format("parquet").saveAsTable("dwd.dim_item")

26/05/06 17:20:26 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

In [ ]:
spark.sql("select distinct behavior from ods.ods_user_behavior").show()

26/05/06 18:00:55 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

+--------+
|behavior|
+--------+
|     fav|
|    cart|
|     buy|
|      pv|
+--------+



AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `event_data` cannot be resolved. Did you mean one of the following? [`event_date`].; line 1 pos 63;
'Sort ['event_data DESC NULLS LAST], true
+- Distinct
   +- Project [event_date#5]
      +- SubqueryAlias spark_catalog.ods.ods_user_behavior
         +- Relation spark_catalog.ods.ods_user_behavior[user_id#0L,item_id#1L,category_id#2L,behavior#3,timestamp#4L,event_date#5] parquet


In [8]:
df_category = spark.table("ods.ods_user_behavior").select("category_id").distinct()
spark.sql("DROP TABLE IF EXISTS dwd.dim_category")
df_category.write.mode("overwrite").format("parquet").saveAsTable("dwd.dim_category")

26/05/06 17:52:55 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

In [15]:
behaviors = [("pv","page_view"),("buy","purchase"),("cart","add_to_cart"),("fav","add_to_favorite")]
from pyspark.sql.types import StructType, StructField, StringType
schema = StructType([
    StructField("behavior", StringType(), True),
    StructField("behavior_name", StringType(), True)
])
df_beh = spark.createDataFrame(behaviors, schema)
df_beh.write.mode("overwrite").format("parquet").saveAsTable("dwd.dim_behavior")


In [18]:
spark.sql("show tables from dwd").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|      dwd|dim_behavior|      false|
|      dwd|dim_category|      false|
|      dwd|    dim_item|      false|
|      dwd|    dim_user|      false|
+---------+------------+-----------+



In [ ]:
df_raw = spark.table("ods.ods_user_behavior").select("event_date").distinct()
df_raw.orderBy("event_date",ascending=False).show()

26/05/06 18:46:02 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

+----------+
|event_date|
+----------+
|2037-04-09|
|2036-10-23|
|2036-10-22|
|2036-10-21|
|2033-03-19|
|2030-01-16|
|2028-10-24|
|2026-10-25|
|2026-10-21|
|2026-10-15|
|2025-11-04|
|2025-11-03|
|2025-11-02|
|2025-11-01|
|2025-10-29|
|2025-10-28|
|2025-10-27|
|2025-10-26|
|2025-10-25|
|2025-10-24|
+----------+
only showing top 20 rows



In [23]:
print(">> Step 3: 构建 DWD 事实表（分桶核心）...")
df_fact = spark.table("ods.ods_user_behavior") \
    .dropDuplicates(["user_id","item_id","category_id","behavior","event_date"]) \
    .filter(
        (F.col("behavior").isin(["pv","buy","cart","fav"])) &
        (F.col("user_id") > 0) &
        (F.col("item_id") > 0) &
        (F.col("category_id") > 0)&
        (F.col("event_date").between("2017-11-25", "2017-12-03"))
    ).select("user_id","item_id","category_id","behavior","timestamp","event_date")


>> Step 3: 构建 DWD 事实表（分桶核心）...


In [26]:
df_fact.show(10)
df_fact.count()

26/05/06 18:50:34 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

+-------+-------+-----------+--------+----------+----------+
|user_id|item_id|category_id|behavior| timestamp|event_date|
+-------+-------+-----------+--------+----------+----------+
| 115590|      9|    3068858|      pv|1511784398|2017-11-27|
| 353414|      9|    3068858|      pv|1512106399|2017-12-01|
| 551746|      9|    3068858|      pv|1512142822|2017-12-01|
| 971591|     11|    1090997|      pv|1511706420|2017-11-26|
| 693226|     14|    4238676|      pv|1512126790|2017-12-01|
| 821520|     14|    4238676|      pv|1511611820|2017-11-25|
| 862304|     14|    4238676|      pv|1511702452|2017-11-26|
| 992812|     14|    4238676|      pv|1512111023|2017-12-01|
| 329487|     15|    4050612|      pv|1511705139|2017-11-26|
| 140418|     19|    2885642|      pv|1512292624|2017-12-03|
+-------+-------+-----------+--------+----------+----------+
only showing top 10 rows



26/05/06 18:51:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/06 18:51:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/06 18:51:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/06 18:51:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/06 18:51:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/06 18:51:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/06 18:51:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/06 18:51:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/06 18:51:27 WARN RowBasedKeyValueBatch: Calling spill() on

87800837

In [27]:
spark.sql("""
    CREATE TABLE dwd.fact_behavior (
        user_id BIGINT, item_id BIGINT, category_id BIGINT,
        behavior STRING, timestamp BIGINT
    ) USING PARQUET
    PARTITIONED BY (event_date STRING)
    CLUSTERED BY (user_id) SORTED BY (item_id) INTO 8 BUCKETS
""")
df_fact.write.mode("overwrite").insertInto("dwd.fact_behavior")

26/05/06 19:08:22 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

In [4]:
print(">> Step 4: 生成 DWS 每日指标...")
spark.sql("CREATE DATABASE IF NOT EXISTS dws")

>> Step 4: 生成 DWS 每日指标...


26/05/06 20:03:54 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/05/06 20:03:54 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/05/06 20:03:55 WARN ObjectStore: Failed to get database dws, returning NoSuchObjectException
26/05/06 20:03:55 WARN ObjectStore: Failed to get database dws, returning NoSuchObjectException
26/05/06 20:03:55 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/05/06 20:03:55 WARN ObjectStore: Failed to get database dws, returning NoSuchObjectException


DataFrame[]

In [5]:
fact = spark.table("dwd.fact_behavior")
df_dws = fact.groupBy("event_date").agg(
    F.count(F.when(F.col("behavior")=="pv",1)).alias("pv"),
    F.countDistinct(F.when(F.col("behavior")=="pv", F.col("user_id"))).alias("uv"),
    F.count(F.when(F.col("behavior")=="buy",1)).alias("buy_count"),
    F.countDistinct(F.when(F.col("behavior")=="buy", F.col("user_id"))).alias("buy_uv"),
    F.count(F.when(F.col("behavior")=="cart",1)).alias("cart_count"),
    F.count(F.when(F.col("behavior")=="fav",1)).alias("fav_count")
).withColumn("pv_to_buy_rate", F.round(F.col("buy_count")/F.col("pv"),6))
df_dws.write.mode("overwrite").partitionBy("event_date") \
    .format("parquet").saveAsTable("dws.dws_behavior_day")

26/05/06 20:23:08 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

In [6]:
print(">> Step 5: 生成 ADS 转化漏斗和 Top10 品类...")
spark.sql("CREATE DATABASE IF NOT EXISTS ads")

>> Step 5: 生成 ADS 转化漏斗和 Top10 品类...


26/05/06 20:26:27 WARN ObjectStore: Failed to get database ads, returning NoSuchObjectException
26/05/06 20:26:27 WARN ObjectStore: Failed to get database ads, returning NoSuchObjectException
26/05/06 20:26:27 WARN ObjectStore: Failed to get database ads, returning NoSuchObjectException


DataFrame[]

In [8]:
import os
WAREHOUSE_DIR = f"/home/lst/my-spark/User_Behavior/spark_warehouse"
os.makedirs(WAREHOUSE_DIR, exist_ok=True)
df_dws.select(
    "event_date",
    F.col("uv").alias("step1_view_users"),
    F.col("fav_count").alias("step2_fav_items"),
    F.col("cart_count").alias("step3_cart_items"),
    F.col("buy_count").alias("step4_buy_items"),
    F.round(F.col("buy_count")/F.col("uv"),6).alias("view_to_buy_rate")
).write.mode("overwrite")\
.partitionBy("event_date")\
.format("parquet") \
.saveAsTable("ads.ads_daily_conversion")

26/05/06 20:45:26 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

In [ ]:
spark.sql("show tables from ads").show()
spark.sql

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|      ads|ads_daily_conversion|      false|
|      ads|    ads_top_category|      false|
+---------+--------------------+-----------+



In [14]:
from pyspark.sql import Window
buy_fact = fact.filter(F.col("behavior")=="buy")
dim_category = spark.table("dwd.dim_category")
df_top = buy_fact.groupBy("event_date","category_id").agg(F.count("*").alias("buy_count"))
window_spec = Window.partitionBy("event_date").orderBy(F.desc("buy_count"))
df_top = df_top.withColumn("rn", F.row_number().over(window_spec)).filter(F.col("rn")<=10)
df_top = df_top.join(dim_category, "category_id", "left").select("event_date","category_id","buy_count","rn")
spark.sql("DROP TABLE IF EXISTS ads.ads_top_category")
df_top.write.mode("overwrite").partitionBy("event_date") \
    .format("parquet").saveAsTable("ads.ads_top_category")
print("All done! 最终数据展示：")
spark.sql("SELECT * FROM ads.ads_daily_conversion LIMIT 5").show(truncate=False)
spark.sql("SELECT * FROM ads.ads_top_category LIMIT 10").show(truncate=False)


26/05/06 20:57:16 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

All done! 最终数据展示：
+----------------+---------------+----------------+---------------+----------------+----------+
|step1_view_users|step2_fav_items|step3_cart_items|step4_buy_items|view_to_buy_rate|event_date|
+----------------+---------------+----------------+---------------+----------------+----------+
|686953          |302071         |557908          |196215         |0.285631        |2017-11-25|
|689260          |291221         |536593          |220468         |0.319862        |2017-11-27|
|688042          |289100         |529119          |206583         |0.300248        |2017-11-28|
|697542          |298587         |546272          |217506         |0.311818        |2017-11-29|
|709586          |302264         |559361          |216337         |0.304878        |2017-11-30|
+----------------+---------------+----------------+---------------+----------------+----------+

+-----------+---------+---+----------+
|category_id|buy_count|rn |event_date|
+-----------+---------+---+----------+


26/05/06 20:57:17 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir